In [57]:
from pathlib import Path
import re

In [58]:
text = Path("../data/fil_ag.md").read_text(encoding="utf-8")

In [59]:
def normalize_markup(line: str) -> str:
    line = line.strip()

    # **## ...** → ## ...
    if line.startswith("**") and line.endswith("**"):
        line = line[2:-2].strip()

    return line

def is_chapter_no(line: str) -> bool:
    line = normalize_markup(line)
    return bool(re.match(r"^## Глава\s+\d+", line))


def is_section(line: str) -> bool:
    line = normalize_markup(line)
    return bool(re.match(r"^## \d+\.\d+\s+", line))


def is_chapter_name(line: str) -> bool:
    line = normalize_markup(line)
    return bool(re.match(r"^## ", line))


def is_definition(line: str) -> bool:
    return bool(
        re.match(r"^∙\s+.*?\bназывается\b", line)
    )

def is_theorem(line: str) -> bool:
    line = normalize_markup(line)
    return bool(re.match(r"^\*\*Теорема\b", line))

def is_proof_start(line: str) -> bool:
    return "♦" in line


def is_proof_end(line: str) -> bool:
    return "⊠" in line

def is_formula_delimiter(line: str) -> bool:
    return line.strip() == "$$"

In [60]:
def classify_line(line: str) -> str:

    if is_chapter_no(line):
        return "chapter_no"

    if is_section(line):
        return "section"

    if is_chapter_name(line):
        return "chapter_name"

    if is_theorem(line):
        return "theorem"

    if is_definition(line):
        return "definition"

    if is_proof_start(line):
        return "proof_start"

    if is_proof_end(line):
        return "proof_end"

    return "text"

In [61]:
def save_current_object(blocks, current_object):
    """
    Добавляет текущий объект в blocks,
    только если объект действительно существует.
    """
    if current_object is not None:
        blocks.append(current_object)


def create_definition(
    chapter,
    chapter_name,
    section,
    text,
):
    return {
        "TYPE": "definition",
        "CHAPTER": chapter,
        "CHAPTER_NAME": chapter_name,
        "SECTION": section,
        "TEXT": text,
    }


def create_theorem(
    chapter,
    chapter_name,
    section,
    statement,
):
    return {
        "TYPE": "theorem",
        "CHAPTER": chapter,
        "CHAPTER_NAME": chapter_name,
        "SECTION": section,
        "STATEMENT": statement,
        "PROOF": [],
    }

In [62]:
blocks = []

current_chapter = ""
current_chapter_name = ""
current_section = ""

current_object = None

mode = "normal"
previous_mode = None
current_formula = []

for line_number, line in enumerate(text.splitlines(), start=1):

    # --------------------------------------------------------
    # Пустые строки
    # --------------------------------------------------------

    if line.strip() == "":
        continue


    # --------------------------------------------------------
    # Определяем тип строки
    # --------------------------------------------------------

    line_type = classify_line(line)


    # ========================================================
    # РЕЖИМ FORMULA
    # ========================================================

    if mode == "formula":

        # Закрывающий $$
        if line_type == "formula_delimiter":

            formula_text = "\n".join(current_formula).strip()

            formula = {
                "TYPE": "formula",
                "TEXT": formula_text,
                "LINE": line_number,
            }

            # Куда положить формулу?
            if (
                current_object is not None
                and current_object["TYPE"] == "theorem"
            ):

                if previous_mode == "theorem_statement":
                    current_object["STATEMENT"].append(formula)

                elif previous_mode == "theorem_proof":
                    current_object["PROOF"].append(formula)

            elif (
                current_object is not None
                and current_object["TYPE"] == "definition"
            ):

                current_object["CONTENT"].append(formula)

            else:

                # Формула встретилась вне объекта
                blocks.append(formula)

            current_formula = []

            # Возвращаемся туда, где были до формулы
            mode = previous_mode
            previous_mode = None

            continue

        # Обычная строка внутри формулы
        current_formula.append(line)

        continue


    # ========================================================
    # CONTEXT: глава
    # ========================================================

    if line_type == "chapter_no":

        current_chapter = re.sub(
            r"^\**##\s*",
            "",
            line.strip()
        )

        continue


    # ========================================================
    # CONTEXT: название главы / темы
    # ========================================================

    if line_type == "chapter_name":

        current_chapter_name = re.sub(
            r"^\**##\s*",
            "",
            line.strip()
        )

        continue


    # ========================================================
    # CONTEXT: раздел
    # ========================================================

    if line_type == "section":

        current_section = re.sub(
            r"^\**##\s*",
            "",
            line.strip()
        )

        continue


    # ========================================================
    # НАЧАЛО FORMULA
    # ========================================================

    if line_type == "formula_delimiter":

        previous_mode = mode
        mode = "formula"
        current_formula = []

        continue


    # ========================================================
    # ТЕОРЕМА
    # ========================================================

    if line_type == "theorem":

        # Сохраняем предыдущий объект
        save_current_object(
            blocks,
            current_object
        )

        # Нормализуем строку
        statement = normalize_markup(line)

        current_object = {
            "TYPE": "theorem",

            "CHAPTER": current_chapter,
            "CHAPTER_NAME": current_chapter_name,
            "SECTION": current_section,

            "STATEMENT": [
                {
                    "TYPE": "text",
                    "TEXT": statement,
                    "LINE": line_number,
                }
            ],

            "PROOF": [],

            "LINE": line_number,
        }

        mode = "theorem_statement"

        # ----------------------------------------------------
        # Важный случай:
        #
        # Теорема и ♦ могут находиться на одной строке.
        # ----------------------------------------------------

        if "♦" in line:

            before_start, after_start = line.split("♦", 1)

            # Перезаписываем statement частью ДО ♦
            current_object["STATEMENT"] = []

            if before_start.strip():

                current_object["STATEMENT"].append({
                    "TYPE": "text",
                    "TEXT": normalize_markup(
                        before_start.strip()
                    ),
                    "LINE": line_number,
                })

            mode = "theorem_proof"

            # Всё после ♦ — начало proof
            if after_start.strip():

                # Может быть даже ♦ ... ⊠
                if "⊠" in after_start:

                    proof_text, after_end = after_start.split(
                        "⊠",
                        1
                    )

                    if proof_text.strip():

                        current_object["PROOF"].append({
                            "TYPE": "text",
                            "TEXT": proof_text.strip(),
                            "LINE": line_number,
                        })

                    save_current_object(
                        blocks,
                        current_object
                    )

                    current_object = None
                    mode = "normal"

                    # Что было после ⊠
                    if after_end.strip():

                        blocks.append({
                            "TYPE": "text",
                            "CHAPTER": current_chapter,
                            "CHAPTER_NAME": current_chapter_name,
                            "SECTION": current_section,
                            "TEXT": after_end.strip(),
                            "LINE": line_number,
                        })

                else:

                    current_object["PROOF"].append({
                        "TYPE": "text",
                        "TEXT": after_start.strip(),
                        "LINE": line_number,
                    })

        continue


    # ========================================================
    # НАЧАЛО PROOF
    # ========================================================

    if (
        line_type == "proof_start"
        and current_object is not None
        and current_object["TYPE"] == "theorem"
        and mode == "theorem_statement"
    ):

        before_start, after_start = line.split("♦", 1)

        # То, что было до ♦, относится к statement
        if before_start.strip():

            current_object["STATEMENT"].append({
                "TYPE": "text",
                "TEXT": before_start.strip(),
                "LINE": line_number,
            })

        mode = "theorem_proof"

        # То, что после ♦, относится к proof
        if after_start.strip():

            # ♦ ... ⊠ на одной строке
            if "⊠" in after_start:

                proof_text, after_end = after_start.split(
                    "⊠",
                    1
                )

                if proof_text.strip():

                    current_object["PROOF"].append({
                        "TYPE": "text",
                        "TEXT": proof_text.strip(),
                        "LINE": line_number,
                    })

                save_current_object(
                    blocks,
                    current_object
                )

                current_object = None
                mode = "normal"

                # Текст после ⊠
                if after_end.strip():

                    blocks.append({
                        "TYPE": "text",
                        "CHAPTER": current_chapter,
                        "CHAPTER_NAME": current_chapter_name,
                        "SECTION": current_section,
                        "TEXT": after_end.strip(),
                        "LINE": line_number,
                    })

        continue


    # ========================================================
    # КОНЕЦ PROOF
    # ========================================================

    if (
        line_type == "proof_end"
        and current_object is not None
        and current_object["TYPE"] == "theorem"
        and mode == "theorem_proof"
    ):

        before_end, after_end = line.split("⊠", 1)

        # Всё до ⊠ добавляется в proof
        if before_end.strip():

            current_object["PROOF"].append({
                "TYPE": "text",
                "TEXT": before_end.strip(),
                "LINE": line_number,
            })

        # Теорема закончена
        save_current_object(
            blocks,
            current_object
        )

        current_object = None
        mode = "normal"

        # Если что-то было после ⊠,
        # это уже не часть theorem
        if after_end.strip():

            blocks.append({
                "TYPE": "text",
                "CHAPTER": current_chapter,
                "CHAPTER_NAME": current_chapter_name,
                "SECTION": current_section,
                "TEXT": after_end.strip(),
                "LINE": line_number,
            })

        continue


    # ========================================================
    # THEOREM STATEMENT
    # ========================================================

    if (
        current_object is not None
        and current_object["TYPE"] == "theorem"
        and mode == "theorem_statement"
    ):

        current_object["STATEMENT"].append({
            "TYPE": "text",
            "TEXT": line.strip(),
            "LINE": line_number,
        })

        continue


    # ========================================================
    # THEOREM PROOF
    # ========================================================

    if (
        current_object is not None
        and current_object["TYPE"] == "theorem"
        and mode == "theorem_proof"
    ):

        current_object["PROOF"].append({
            "TYPE": "text",
            "TEXT": line.strip(),
            "LINE": line_number,
        })

        continue


    # ========================================================
    # НОВОЕ DEFINITION
    # ========================================================

    if line_type == "definition":

        # Сохраняем старый объект
        save_current_object(
            blocks,
            current_object
        )

        # Создаём новый definition
        current_object = {
            "TYPE": "definition",

            "CHAPTER": current_chapter,
            "CHAPTER_NAME": current_chapter_name,
            "SECTION": current_section,

            "CONTENT": [
                {
                    "TYPE": "text",
                    "TEXT": line.strip(),
                    "LINE": line_number,
                }
            ],

            "LINE": line_number,
        }

        mode = "normal"

        continue


    # ========================================================
    # ОБЫЧНЫЙ ТЕКСТ ВНУТРИ DEFINITION
    # ========================================================

    if (
        current_object is not None
        and current_object["TYPE"] == "definition"
    ):

        current_object["CONTENT"].append({
            "TYPE": "text",
            "TEXT": line.strip(),
            "LINE": line_number,
        })

        continue


    # ========================================================
    # ОБЫЧНЫЙ ТЕКСТ ВНЕ ОБЪЕКТА
    # ========================================================

    blocks.append({
        "TYPE": "text",

        "CHAPTER": current_chapter,
        "CHAPTER_NAME": current_chapter_name,
        "SECTION": current_section,

        "TEXT": line.strip(),

        "LINE": line_number,
    })


# ============================================================
# Сохраняем последний объект
# ============================================================

save_current_object(
    blocks,
    current_object
)


In [63]:
for block in blocks[:30]:
    print(block)

{'TYPE': 'definition', 'CHAPTER': 'Глава 1', 'CHAPTER_NAME': 'Метод координат', 'SECTION': '1.1 Величина направленного отрезка. Теорема Шаля. Декартова система координат на прямой.', 'CONTENT': [{'TYPE': 'text', 'TEXT': '∙ Отрезком называется часть прямой, ограниченная двумя точками. Отрезок называется направленным, если указано, какая из граничных точек является начальной и какая конечной (обозначения: $\\overline{AB}$ — направленный отрезок; $|\\overline{AB}|$ — длина направленного отрезка).', 'LINE': 7}], 'LINE': 7}
{'TYPE': 'definition', 'CHAPTER': 'Глава 1', 'CHAPTER_NAME': 'Метод координат', 'SECTION': '1.1 Величина направленного отрезка. Теорема Шаля. Декартова система координат на прямой.', 'CONTENT': [{'TYPE': 'text', 'TEXT': '∙ Отрезок называется нулевым, если его начальная и конечная точки совпадают. Длина нулевого направленного отрезка равна нулю.', 'LINE': 9}], 'LINE': 9}
{'TYPE': 'definition', 'CHAPTER': 'Глава 1', 'CHAPTER_NAME': 'Метод координат', 'SECTION': '1.1 Величи